In [ ]:
import pandas as pd
import time

from ta.volatility import AverageTrueRange
from backtesting import Backtest

global_range_to_consider = 0
global_tp_multiple = 0
global_stop_trade_hour = 0
global_close_range_hour = 0

class OpeningRangeBreakout(Strategy):
    # variables
    range_to_consider = 3
    close_range_hour = 9
    tp_multiple = 3
    stop_trade_hour = 20

    # fixed variables
    risk_percent = 0.01
    max_leverage = 4
    my_equity = 10000
    open_range_hour = close_range_hour - range_to_consider

    # def init(self, range_to_consider, tp_multiple, stop_trade_hour, close_range_hour):
    def init(self):
        self.current_day = None
        self.opening_range_high = None
        self.opening_range_low = None
        if global_range_to_consider != 0:
            print(global_range_to_consider)
            self.range_to_consider = global_range_to_consider

        if global_tp_multiple != 0:
            self.tp_multiple = global_tp_multiple

        if global_stop_trade_hour != 0:
            self.stop_trade_hour = global_stop_trade_hour

        if global_close_range_hour != 0:
            self.close_range_hour = global_close_range_hour

        if global_close_range_hour != 0 and global_range_to_consider != 0:
            self.open_range_hour = global_close_range_hour - global_range_to_consider

    def reset_range(self, day):
        self.current_day = day
        self.current_day_position_taked = False
        self.opening_range_high = None
        self.opening_range_low = None

    def get_position_size(self, entry_price: float, stop_price : float) -> int:
        per_share_risk = abs(entry_price - stop_price)
        print(per_share_risk)
        if per_share_risk == 0:
            return 0

        share_by_risk = (self.risk_percent * self.equity) / entry_price

        share_by_leverage = (self.max_leverage * self.equity) / entry_price
        print(f"share_by_risk = {share_by_risk} & share_by_leverage = {share_by_leverage}")

        return int(min(share_by_risk, share_by_leverage))

    def next(self):
        t = self.data.index[-1]
        current_bar_date = self.data.index[-1].date()

        if (self.current_day != current_bar_date):
            self.reset_range(current_bar_date)
            # print(f"new date {current_bar_date}")

        if self.open_range_hour < t.time().hour and t.time().hour <= self.close_range_hour:
            if self.opening_range_high is not None:
                self.opening_range_high = max(self.opening_range_high, self.data.High[-1])
            else:
                self.opening_range_high = self.data.High[-1]

            if self.opening_range_low is not None:
                    self.opening_range_low = min(self.opening_range_low, self.data.Low[-1])
            else:
                self.opening_range_low = self.data.Low[-1]

            if t.time().hour < self.close_range_hour:
                return

        if t.time().hour > self.close_range_hour and self.opening_range_high is not None and self.opening_range_low is not None:
            c_long = self.data.Open[-1] < self.opening_range_high and self.data.Close[-1] > self.opening_range_high
            c_short = self.data.Open[-1] > self.opening_range_low and self.data.Close[-1] < self.opening_range_low
            
            c2 = (self.opening_range_high - self.opening_range_low) < self.data['ATR'][-1] * 2.5
            c3 = (self.opening_range_high - self.opening_range_low) > self.data['ATR'][-1] * 1.0
            c4 = t.time().hour < self.stop_trade_hour

            if not self.current_day_position_taked:
                # if c_long and c2 and c3 and c4:
                #     planned_entry_price = self.data.Close[-1]
                #     sl_price = self.opening_range_low
                #     per_share_risk = abs(planned_entry_price - sl_price)
                #     tp_price = planned_entry_price + per_share_risk * self.tp_multiple
                #     position_size = int((self.risk_percent * self.my_equity) / per_share_risk)

                #     # print(f'Go long tp = {tp_price} sl = {sl_price} position size = {position_size}')
                #     order = self.buy(size = position_size, tp=tp_price, sl=sl_price)
                #     self.current_day_position_taked = True
                if c_short and c2 and c3 and c4:
                    planned_entry_price = self.data.Close[-1]
                    sl_price = self.opening_range_high
                    per_share_risk = abs(planned_entry_price - sl_price)
                    tp_price = planned_entry_price - per_share_risk * self.tp_multiple
                    position_size = int((self.risk_percent * self.my_equity) / per_share_risk)

                    # print(f'Go short tp = {tp_price} sl = {sl_price} position size = {position_size}')
                    order = self.sell(size = position_size, tp=tp_price, sl=sl_price)
                    self.current_day_position_taked = True


df = pd.read_csv("./Trading Data/CADCHF_H1.csv",sep=',', parse_dates=['Date'], index_col='Date')
df = df.loc['2022-05-01' : '2026-9-16']

indicator_atr = AverageTrueRange(df['High'], df['Low'], df['Close'], window=14)
df['ATR'] = indicator_atr._atr

bt = Backtest(df, OpeningRangeBreakout, margin=0.02, finalize_trades=True)

stats = bt.run()

print(stats)

bt.plot()

c:\Users\kyanh\AppData\Local\Programs\Python\Python313-arm64\Lib\site-packages\backtesting\backtesting.py:1009: UserWarning: time=25497: Broker canceled the order due to insufficient margin (equity=3409.66, margin_available=1407.21).
  warnings.warn(
c:\Users\kyanh\AppData\Local\Programs\Python\Python313-arm64\Lib\site-packages\backtesting\backtesting.py:1009: UserWarning: time=26676: Broker canceled the order due to insufficient margin (equity=1666.03, margin_available=827.28).
  warnings.warn(
c:\Users\kyanh\AppData\Local\Programs\Python\Python313-arm64\Lib\site-packages\backtesting\backtesting.py:1009: UserWarning: time=26935: Broker canceled the order due to insufficient margin (equity=1331.31, margin_available=1331.31).
  warnings.warn(
c:\Users\kyanh\AppData\Local\Programs\Python\Python313-arm64\Lib\site-packages\backtesting\backtesting.py:1009: UserWarning: time=27083: Broker canceled the order due to insufficient margin (equity=1083.25, margin_available=160.80).
  warnings.warn

Start                     2022-05-01 21:00:00
End                       2026-09-16 23:00:00
Duration                   1599 days 02:00:00
Exposure Time [%]                    63.40633
Equity Final [$]                    780.06161
Equity Peak [$]                   13965.40249
Return [%]                          -92.19938
Buy & Hold Return [%]               -22.01131
Return (Ann.) [%]                   -37.47301
Volatility (Ann.) [%]                22.42489
CAGR [%]                            -44.15956
Sharpe Ratio                         -1.67105
Sortino Ratio                        -1.47411
Calmar Ratio                          -0.3969
Alpha [%]                           -96.79857
Beta                                 -0.20895
Max. Drawdown [%]                   -94.41433
Avg. Drawdown [%]                    -2.67592
Max. Drawdown Duration     1034 days 11:00:00
Avg. Drawdown Duration       23 days 15:00:00
# Trades                                  880
Win Rate [%]                      

GridPlot(id='p3249', ...)

In [ ]:
df = df.loc['2022-05-01' : '2026-9-16']

range_to_consider_list = [3]
close_range_hour_list = [7,9]
stop_trade_hour_list = [19,20,21]
tp_multiple_list = [2,2.5,3,3.5,4,4.5,5,5.5,6]


with open(f"./results/result_CADCHF_{time.time()}.txt", "a") as f:
    for range in range_to_consider_list:
        for close_range_hour in close_range_hour_list:
             for stop_trade_hour in stop_trade_hour_list:
                for tp_multiple in tp_multiple_list:
                    # range_to_consider, tp_multiple, stop_trade_hour, close_range_hour
                    global_range_to_consider = range
                    global_close_range_hour = close_range_hour
                    global_stop_trade_hour = stop_trade_hour
                    global_tp_multiple = tp_multiple

                    bt = Backtest(df, OpeningRangeBreakout, margin=0.02, finalize_trades=True)
                    stats = bt.run()

                    f.write(f"range to consider = {range} & close range hour = {close_range_hour} & stop_trade_hour = {stop_trade_hour} & tp_multiple = {tp_multiple}\n")
                    f.write(f"Return [%] = {stats['Return [%]']} | Return (Ann.) [%] = {stats['Return (Ann.) [%]']} | Max. Drawdown [%] = {stats['Max. Drawdown [%]']} | Win Rate [%] = {stats['Win Rate [%]']}\n")
                    f.write("----------------------------------------------\n")

                    print(f"range to consider = {range} & close range hour = {close_range_hour} & stop_trade_hour = {stop_trade_hour} & tp_multiple = {tp_multiple} DONE")



5
range to consider = 5 & close range hour = 7 & stop_trade_hour = 19 & tp_multiple = 2 DONE
5
range to consider = 5 & close range hour = 7 & stop_trade_hour = 19 & tp_multiple = 2.5 DONE
5
range to consider = 5 & close range hour = 7 & stop_trade_hour = 19 & tp_multiple = 3 DONE
5
range to consider = 5 & close range hour = 7 & stop_trade_hour = 19 & tp_multiple = 3.5 DONE
5
range to consider = 5 & close range hour = 7 & stop_trade_hour = 19 & tp_multiple = 4 DONE
5
range to consider = 5 & close range hour = 7 & stop_trade_hour = 19 & tp_multiple = 4.5 DONE
5
range to consider = 5 & close range hour = 7 & stop_trade_hour = 19 & tp_multiple = 5 DONE
5
range to consider = 5 & close range hour = 7 & stop_trade_hour = 19 & tp_multiple = 5.5 DONE
5
range to consider = 5 & close range hour = 7 & stop_trade_hour = 19 & tp_multiple = 6 DONE
5
range to consider = 5 & close range hour = 7 & stop_trade_hour = 20 & tp_multiple = 2 DONE
5
range to consider = 5 & close range hour = 7 & stop_trade_ho

In [ ]:
import pandas as pd
import time

from ta.volatility import AverageTrueRange
from backtesting import Backtest

global_range_to_consider = 0
global_tp_multiple = 0
global_stop_trade_hour = 0
global_close_range_hour = 0

class OpeningRangeBreakout(Strategy):
    # variables
    range_to_consider = 5
    close_range_hour = 8
    tp_multiple = 1.5
    stop_trade_hour = 18

    # fixed variables
    risk_percent = 0.005
    max_leverage = 4
    my_equity = 20000
    open_range_hour = close_range_hour - range_to_consider

    # def init(self, range_to_consider, tp_multiple, stop_trade_hour, close_range_hour):
    def init(self):
        self.current_day = None
        self.opening_range_high = None
        self.opening_range_low = None
        if global_range_to_consider != 0:
            self.range_to_consider = global_range_to_consider

        if global_tp_multiple != 0:
            self.tp_multiple = global_tp_multiple

        if global_stop_trade_hour != 0:
            self.stop_trade_hour = global_stop_trade_hour

        if global_close_range_hour != 0:
            self.close_range_hour = global_close_range_hour

        if global_close_range_hour != 0 and global_range_to_consider != 0:
            self.open_range_hour = global_close_range_hour - global_range_to_consider

    def reset_range(self, day):
        self.current_day = day
        self.current_day_position_taked = False
        self.opening_range_high = None
        self.opening_range_low = None

    def get_position_size(self, entry_price: float, stop_price : float) -> int:
        per_share_risk = abs(entry_price - stop_price)
        print(per_share_risk)
        if per_share_risk == 0:
            return 0

        share_by_risk = (self.risk_percent * self.equity) / entry_price

        share_by_leverage = (self.max_leverage * self.equity) / entry_price
        print(f"share_by_risk = {share_by_risk} & share_by_leverage = {share_by_leverage}")

        return int(min(share_by_risk, share_by_leverage))

    def next(self):
        t = self.data.index[-1]
        current_bar_date = self.data.index[-1].date()

        if (self.current_day != current_bar_date):
            self.reset_range(current_bar_date)
            # print(f"new date {current_bar_date}")

        if self.open_range_hour < t.time().hour and t.time().hour <= self.close_range_hour:
            if self.opening_range_high is not None:
                self.opening_range_high = max(self.opening_range_high, self.data.High[-1])
            else:
                self.opening_range_high = self.data.High[-1]

            if self.opening_range_low is not None:
                    self.opening_range_low = min(self.opening_range_low, self.data.Low[-1])
            else:
                self.opening_range_low = self.data.Low[-1]

            if t.time().hour < self.close_range_hour:
                return

        if t.time().hour > self.close_range_hour and self.opening_range_high is not None and self.opening_range_low is not None:
            c_long = self.data.Open[-1] < self.opening_range_high and self.data.Close[-1] > self.opening_range_high
            c_short = self.data.Open[-1] > self.opening_range_low and self.data.Close[-1] < self.opening_range_low
            
            c2 = (self.opening_range_high - self.opening_range_low) < self.data['ATR'][-1] * 2.5
            c3 = (self.opening_range_high - self.opening_range_low) > self.data['ATR'][-1] * 1.0
            c4 = t.time().hour < self.stop_trade_hour

            if not self.current_day_position_taked:
                if c_long and c2 and c3 and c4:
                    planned_entry_price = self.data.Close[-1]
                    sl_price = self.opening_range_low
                    per_share_risk = abs(planned_entry_price - sl_price)
                    tp_price = planned_entry_price + per_share_risk * self.tp_multiple
                    position_size = int((self.risk_percent * self.my_equity) / per_share_risk)

                    # print(f'Go long tp = {tp_price} sl = {sl_price} position size = {position_size}')
                    order = self.buy(size = position_size, tp=tp_price, sl=sl_price)
                    self.current_day_position_taked = True
                if c_short and c2 and c3 and c4:
                    planned_entry_price = self.data.Close[-1]
                    sl_price = self.opening_range_high
                    per_share_risk = abs(planned_entry_price - sl_price)
                    tp_price = planned_entry_price - per_share_risk * self.tp_multiple
                    position_size = int((self.risk_percent * self.my_equity) / per_share_risk)

                    # print(f'Go short tp = {tp_price} sl = {sl_price} position size = {position_size}')
                    order = self.sell(size = position_size, tp=tp_price, sl=sl_price)
                    self.current_day_position_taked = True


df = pd.read_csv("./Trading Data/CADCHF_H1.csv",sep=',', parse_dates=['Date'], index_col='Date')
df = df.loc['2020-01-01' : '2026-9-16']

indicator_atr = AverageTrueRange(df['High'], df['Low'], df['Close'], window=14)
df['ATR'] = indicator_atr._atr
result = []

range_to_consider_list = [3,4,5]
close_range_hour_list = [7,8,9]
stop_trade_hour_list = [18,19,20,21]
tp_multiple_list = [2,2.5,3,3.5,4,4.5,5,5.5,6]

with open(f"./results/result_USDCAD_{time.time()}.txt", "a") as f:
    for range in range_to_consider_list:
        for close_range_hour in close_range_hour_list:
            for stop_trade_hour in stop_trade_hour_list:
                for tp_multiple in tp_multiple_list:
                    # range_to_consider, tp_multiple, stop_trade_hour, close_range_hour
                    global_range_to_consider = range
                    global_close_range_hour = close_range_hour
                    global_stop_trade_hour = stop_trade_hour
                    global_tp_multiple = tp_multiple


                    bt = Backtest(df, OpeningRangeBreakout, margin=0.02, finalize_trades=True)
                    stats = bt.run()

                    f.write(f"range to consider = {range} & close range hour = {close_range_hour} & stop_trade_hour = {stop_trade_hour} & tp_multiple = {tp_multiple}\n")
                    f.write(f"Return [%] = {stats['Return [%]']} | Return (Ann.) [%] = {stats['Return (Ann.) [%]']} | Max. Drawdown [%] = {stats['Max. Drawdown [%]']} | Win Rate [%] = {stats['Win Rate [%]']}\n")
                    f.write("----------------------------------------------\n")

                    print(f"range to consider = {range} & close range hour = {close_range_hour} & stop_trade_hour = {stop_trade_hour} & tp_multiple = {tp_multiple} DONE")



# Return [%] Return (Ann.) [%] Max. Drawdown [%]
result


range to consider = 3 & close range hour = 7 & stop_trade_hour = 18 & tp_multiple = 2 DONE


c:\Users\kyanh\AppData\Local\Programs\Python\Python313-arm64\Lib\site-packages\backtesting\backtesting.py:1009: UserWarning: time=23112: Broker canceled the order due to insufficient margin (equity=2711.67, margin_available=458.51).
  warnings.warn(
c:\Users\kyanh\AppData\Local\Programs\Python\Python313-arm64\Lib\site-packages\backtesting\backtesting.py:1009: UserWarning: time=23138: Broker canceled the order due to insufficient margin (equity=3033.41, margin_available=786.68).
  warnings.warn(
c:\Users\kyanh\AppData\Local\Programs\Python\Python313-arm64\Lib\site-packages\backtesting\backtesting.py:1009: UserWarning: time=27872: Broker canceled the order due to insufficient margin (equity=3925.64, margin_available=1749.31).
  warnings.warn(
c:\Users\kyanh\AppData\Local\Programs\Python\Python313-arm64\Lib\site-packages\backtesting\backtesting.py:1009: UserWarning: time=28232: Broker canceled the order due to insufficient margin (equity=4281.98, margin_available=452.01).
  warnings.warn(

range to consider = 3 & close range hour = 7 & stop_trade_hour = 18 & tp_multiple = 2.5 DONE


c:\Users\kyanh\AppData\Local\Programs\Python\Python313-arm64\Lib\site-packages\backtesting\backtesting.py:1009: UserWarning: time=23112: Broker canceled the order due to insufficient margin (equity=3111.67, margin_available=858.50).
  warnings.warn(
c:\Users\kyanh\AppData\Local\Programs\Python\Python313-arm64\Lib\site-packages\backtesting\backtesting.py:1009: UserWarning: time=28232: Broker canceled the order due to insufficient margin (equity=5049.72, margin_available=1219.74).
  warnings.warn(
c:\Users\kyanh\AppData\Local\Programs\Python\Python313-arm64\Lib\site-packages\backtesting\backtesting.py:1009: UserWarning: time=31703: Broker canceled the order due to insufficient margin (equity=2690.01, margin_available=1513.45).
  warnings.warn(


range to consider = 3 & close range hour = 7 & stop_trade_hour = 18 & tp_multiple = 3 DONE
range to consider = 3 & close range hour = 7 & stop_trade_hour = 18 & tp_multiple = 3.5 DONE


...

range to consider = 5 & close range hour = 9 & stop_trade_hour = 21 & tp_multiple = 5.5 DONE


c:\Users\kyanh\AppData\Local\Programs\Python\Python313-arm64\Lib\site-packages\backtesting\backtesting.py:1009: UserWarning: time=18631: Broker canceled the order due to insufficient margin (equity=1690.18, margin_available=1363.65).
  warnings.warn(
c:\Users\kyanh\AppData\Local\Programs\Python\Python313-arm64\Lib\site-packages\backtesting\backtesting.py:1009: UserWarning: time=20118: Broker canceled the order due to insufficient margin (equity=520.30, margin_available=520.30).
  warnings.warn(
c:\Users\kyanh\AppData\Local\Programs\Python\Python313-arm64\Lib\site-packages\backtesting\backtesting.py:1009: UserWarning: time=20138: Broker canceled the order due to insufficient margin (equity=520.30, margin_available=520.30).
  warnings.warn(
c:\Users\kyanh\AppData\Local\Programs\Python\Python313-arm64\Lib\site-packages\backtesting\backtesting.py:1009: UserWarning: time=20237: Broker canceled the order due to insufficient margin (equity=520.30, margin_available=520.30).
  warnings.warn(
c:

range to consider = 5 & close range hour = 9 & stop_trade_hour = 21 & tp_multiple = 6 DONE


c:\Users\kyanh\AppData\Local\Programs\Python\Python313-arm64\Lib\site-packages\backtesting\backtesting.py:1009: UserWarning: time=39544: Broker canceled the order due to insufficient margin (equity=769.82, margin_available=162.62).
  warnings.warn(
c:\Users\kyanh\AppData\Local\Programs\Python\Python313-arm64\Lib\site-packages\backtesting\backtesting.py:1009: UserWarning: time=39660: Broker canceled the order due to insufficient margin (equity=581.25, margin_available=393.67).
  warnings.warn(
c:\Users\kyanh\AppData\Local\Programs\Python\Python313-arm64\Lib\site-packages\backtesting\backtesting.py:1009: UserWarning: time=39798: Broker canceled the order due to insufficient margin (equity=544.88, margin_available=356.58).
  warnings.warn(
c:\Users\kyanh\AppData\Local\Programs\Python\Python313-arm64\Lib\site-packages\backtesting\backtesting.py:1009: UserWarning: time=39870: Broker canceled the order due to insufficient margin (equity=510.91, margin_available=321.92).
  warnings.warn(
c:\U

[]

In [54]:
import numpy as np
import pandas as pd
import time

from ta.trend import EMAIndicator
from ta.volatility import AverageTrueRange
from backtesting import Backtest

global_range_to_consider = 0
global_tp_multiple = 0
global_stop_trade_hour = 0
global_close_range_hour = 0

class OpeningRangeBreakout(Strategy):
    # variables
    range_to_consider = 3
    close_range_hour = 9
    tp_multiple = 3
    stop_trade_hour = 20

    # fixed variables
    risk_percent = 0.01
    max_leverage = 4
    my_equity = 10000
    open_range_hour = close_range_hour - range_to_consider

    # def init(self, range_to_consider, tp_multiple, stop_trade_hour, close_range_hour):
    def init(self):
        self.current_day = None
        self.opening_range_high = None
        self.opening_range_low = None
        if global_range_to_consider != 0:
            print(global_range_to_consider)
            self.range_to_consider = global_range_to_consider

        if global_tp_multiple != 0:
            self.tp_multiple = global_tp_multiple

        if global_stop_trade_hour != 0:
            self.stop_trade_hour = global_stop_trade_hour

        if global_close_range_hour != 0:
            self.close_range_hour = global_close_range_hour

        if global_close_range_hour != 0 and global_range_to_consider != 0:
            self.open_range_hour = global_close_range_hour - global_range_to_consider

    def reset_range(self, day):
        self.current_day = day
        self.current_day_position_taked = False
        self.opening_range_high = None
        self.opening_range_low = None

    def get_position_size(self, entry_price: float, stop_price : float) -> int:
        per_share_risk = abs(entry_price - stop_price)
        print(per_share_risk)
        if per_share_risk == 0:
            return 0

        share_by_risk = (self.risk_percent * self.equity) / entry_price

        share_by_leverage = (self.max_leverage * self.equity) / entry_price
        print(f"share_by_risk = {share_by_risk} & share_by_leverage = {share_by_leverage}")

        return int(min(share_by_risk, share_by_leverage))

    def next(self):
        t = self.data.index[-1]
        current_bar_date = self.data.index[-1].date()

        if (self.current_day != current_bar_date):
            self.reset_range(current_bar_date)
            # print(f"new date {current_bar_date}")

        if self.open_range_hour < t.time().hour and t.time().hour <= self.close_range_hour:
            if self.opening_range_high is not None:
                self.opening_range_high = max(self.opening_range_high, self.data.High[-1])
            else:
                self.opening_range_high = self.data.High[-1]

            if self.opening_range_low is not None:
                    self.opening_range_low = min(self.opening_range_low, self.data.Low[-1])
            else:
                self.opening_range_low = self.data.Low[-1]

            if t.time().hour < self.close_range_hour:
                return

        if t.time().hour > self.close_range_hour and self.opening_range_high is not None and self.opening_range_low is not None:
            c_long = self.data.Open[-1] < self.opening_range_high and self.data.Close[-1] > self.opening_range_high and self.data['dl_direction'][-1] == 1
            c_short = self.data.Open[-1] > self.opening_range_low and self.data.Close[-1] < self.opening_range_low and self.data['dl_direction'][-1] == 2
            
            c2 = (self.opening_range_high - self.opening_range_low) < self.data['ATR'][-1] * 2.5
            c3 = (self.opening_range_high - self.opening_range_low) > self.data['ATR'][-1] * 1.0
            c4 = t.time().hour < self.stop_trade_hour

            if not self.current_day_position_taked:
                # if c_long and c2 and c3 and c4:
                #     planned_entry_price = self.data.Close[-1]
                #     sl_price = self.opening_range_low
                #     per_share_risk = abs(planned_entry_price - sl_price)
                #     tp_price = planned_entry_price + per_share_risk * self.tp_multiple
                #     position_size = int((self.risk_percent * self.my_equity) / per_share_risk)

                #     # print(f'Go long tp = {tp_price} sl = {sl_price} position size = {position_size}')
                #     order = self.buy(size = position_size, tp=tp_price, sl=sl_price)
                #     self.current_day_position_taked = True
                if c_short and c2 and c3 and c4:
                    planned_entry_price = self.data.Close[-1]
                    sl_price = self.opening_range_high
                    per_share_risk = abs(planned_entry_price - sl_price)
                    tp_price = planned_entry_price - per_share_risk * self.tp_multiple
                    position_size = int((self.risk_percent * self.my_equity) / per_share_risk)

                    # print(f'Go short tp = {tp_price} sl = {sl_price} position size = {position_size}')
                    order = self.sell(size = position_size, tp=tp_price, sl=sl_price)
                    self.current_day_position_taked = True

start_date = '2022-1-1'
end_date = '2026-9-16'

df_d1 = pd.read_csv("./Trading Data/CADCHF_D1.csv",sep=',', parse_dates=['Date'], index_col='Date')
df_d1 = df_d1.loc[start_date : end_date]

indicator_ema_5 = EMAIndicator(df_d1['Close'], 5)
indicator_ema_8 = EMAIndicator(df_d1['Close'], 8)
df_d1['EMA5'] = indicator_ema_5.ema_indicator();
df_d1['EMA8'] = indicator_ema_8.ema_indicator();

df_h1 = pd.read_csv("./Trading Data/CADCHF_H1.csv",sep=',', parse_dates=['Date'], index_col='Date')
df_h1 = df_h1.loc[start_date : end_date]

previous_row = None
previous_direction = 0
d1_tf_direction_list = []

for index, row in df_d1.iterrows():
    df_by_day = df_h1.loc[f'{index.date()}' : f'{index.date()}']

    if index.weekday() == 5 or index.weekday() == 6:
        for index2, row2 in df_by_day.iterrows():
            d1_tf_direction_list.append(0)
        continue
    else:
        for index2, row2 in df_by_day.iterrows():
            d1_tf_direction_list.append(previous_direction)
    
    if not np.isnan(row['EMA5']) and not np.isnan(row['EMA8']):
        if previous_row is not None:
            if previous_row['EMA5'] < previous_row['EMA8'] and row['EMA5'] > row['EMA8']:
                # print(f"{index} crossed BULLISH ema5 = {row['EMA5']} & ema8 = {row['EMA8']}")
                previous_direction = 1

            if previous_row['EMA5'] > previous_row['EMA8'] and row['EMA5'] < row['EMA8']:
                # print(f"{index} crossed BEARISH ema5 = {row['EMA5']} & ema8 = {row['EMA8']}")
                previous_direction = 2

        previous_row = row

df_h1['dl_direction'] = d1_tf_direction_list
indicator_atr = AverageTrueRange(df_h1['High'], df_h1['Low'], df_h1['Close'], window=14)
df_h1['ATR'] = indicator_atr._atr

bt = Backtest(df_h1, OpeningRangeBreakout, margin=0.02, finalize_trades=True)

stats = bt.run()

print(stats)

bt.plot()

Start                     2022-01-02 22:00:00
End                       2026-09-16 23:00:00
Duration                   1718 days 01:00:00
Exposure Time [%]                    30.60856
Equity Final [$]                  18053.97582
Equity Peak [$]                   18801.78087
Return [%]                           80.53976
Buy & Hold Return [%]                -18.0362
Return (Ann.) [%]                    10.65066
Volatility (Ann.) [%]                13.20322
CAGR [%]                             13.38263
Sharpe Ratio                          0.80667
Sortino Ratio                         1.74063
Calmar Ratio                          0.65977
Alpha [%]                            67.52629
Beta                                 -0.72152
Max. Drawdown [%]                   -16.14287
Avg. Drawdown [%]                    -1.34719
Max. Drawdown Duration      255 days 22:00:00
Avg. Drawdown Duration       12 days 05:00:00
# Trades                                  282
Win Rate [%]                      

c:\Users\kyanh\AppData\Local\Programs\Python\Python313-arm64\Lib\site-packages\backtesting\_plotting.py:141: UserWarning: Data contains too many candlesticks to plot; downsampling to '8h'. See `Backtest.plot(resample=...)`
  warnings.warn(f"Data contains too many candlesticks to plot; downsampling to {freq!r}. "


GridPlot(id='p6089', ...)

In [18]:
import pandas as pd
import numpy as np
from ta.trend import EMAIndicator
from ta.volatility import AverageTrueRange
from backtesting import Backtest, Strategy

timezone = "Europe/London"

start_date = '2020-1-1'
end_date = '2026-9-16'

df_d1 = pd.read_csv("./Trading Data/USDCAD_D1.csv",sep=',', parse_dates=['Date'], index_col='Date')
df_d1 = df_d1.loc[start_date : end_date]

indicator_ema_5 = EMAIndicator(df_d1['Close'], 5)
indicator_ema_8 = EMAIndicator(df_d1['Close'], 8)
df_d1['EMA5'] = indicator_ema_5.ema_indicator();
df_d1['EMA8'] = indicator_ema_8.ema_indicator();

df_h1 = pd.read_csv("./Trading Data/USDCAD_H1.csv",sep=',', parse_dates=['Date'], index_col='Date')
df_h1 = df_h1.loc[start_date : end_date]

previous_row = None
previous_direction = 0
d1_tf_direction_list = []

for index, row in df_d1.iterrows():
    df_by_day = df_h1.loc[f'{index.date()}' : f'{index.date()}']

    if index.weekday() == 5 or index.weekday() == 6:
        for index2, row2 in df_by_day.iterrows():
            d1_tf_direction_list.append(0)
        continue
    else:
        for index2, row2 in df_by_day.iterrows():
            d1_tf_direction_list.append(previous_direction)
    
    if not np.isnan(row['EMA5']) and not np.isnan(row['EMA8']):
        if previous_row is not None:
            if previous_row['EMA5'] < previous_row['EMA8'] and row['EMA5'] > row['EMA8']:
                # print(f"{index} crossed BULLISH ema5 = {row['EMA5']} & ema8 = {row['EMA8']}")
                previous_direction = 1

            if previous_row['EMA5'] > previous_row['EMA8'] and row['EMA5'] < row['EMA8']:
                # print(f"{index} crossed BEARISH ema5 = {row['EMA5']} & ema8 = {row['EMA8']}")
                previous_direction = 2

        previous_row = row

df_h1['dl_direction'] = d1_tf_direction_list
indicator_atr = AverageTrueRange(df_h1['High'], df_h1['Low'], df_h1['Close'], window=14)
df_h1['ATR'] = indicator_atr._atr
df_h1.index = pd.to_datetime(df_h1.index, utc=True).tz_convert(timezone)

range_to_consider_list = [5]
close_range_hour_list = [6,7,8,9]
stop_trade_hour_list = [18,19,20,21]
tp_multiple_list = [1,1.5,2,2.5,3,3.5,4,4.5,5,5.5,6]

with open(f"./results/result_USDCAD_{time.time()}.txt", "a") as f:
    for range in range_to_consider_list:
        for close_range_hour in close_range_hour_list:
            for stop_trade_hour in stop_trade_hour_list:
                for tp_multiple in tp_multiple_list:
                    # range_to_consider, tp_multiple, stop_trade_hour, close_range_hour
                    global_range_to_consider = range
                    global_close_range_hour = close_range_hour
                    global_stop_trade_hour = stop_trade_hour
                    global_tp_multiple = tp_multiple

                    bt = Backtest(df_h1, OpeningRangeBreakout, margin=0.02, finalize_trades=True)
                    stats = bt.run()

                    f.write(f"range to consider = {range} & close range hour = {close_range_hour} & stop_trade_hour = {stop_trade_hour} & tp_multiple = {tp_multiple}\n")
                    f.write(f"Return [%] = {stats['Return [%]']} | Return (Ann.) [%] = {stats['Return (Ann.) [%]']} | Max. Drawdown [%] = {stats['Max. Drawdown [%]']} | Win Rate [%] = {stats['Win Rate [%]']}\n")
                    f.write("----------------------------------------------\n")

                    print(f"range to consider = {range} & close range hour = {close_range_hour} & stop_trade_hour = {stop_trade_hour} & tp_multiple = {tp_multiple} DONE")


range to consider = 5 & close range hour = 6 & stop_trade_hour = 18 & tp_multiple = 1 DONE
range to consider = 5 & close range hour = 6 & stop_trade_hour = 18 & tp_multiple = 1.5 DONE
range to consider = 5 & close range hour = 6 & stop_trade_hour = 18 & tp_multiple = 2 DONE


c:\Users\kyanh\AppData\Local\Programs\Python\Python313-arm64\Lib\site-packages\backtesting\backtesting.py:1009: UserWarning: time=39910: Broker canceled the order due to insufficient margin (equity=5071.06, margin_available=2172.64).
  warnings.warn(


range to consider = 5 & close range hour = 6 & stop_trade_hour = 18 & tp_multiple = 2.5 DONE
range to consider = 5 & close range hour = 6 & stop_trade_hour = 18 & tp_multiple = 3 DONE
range to consider = 5 & close range hour = 6 & stop_trade_hour = 18 & tp_multiple = 3.5 DONE
range to consider = 5 & close range hour = 6 & stop_trade_hour = 18 & tp_multiple = 4 DONE
range to consider = 5 & close range hour = 6 & stop_trade_hour = 18 & tp_multiple = 4.5 DONE
range to consider = 5 & close range hour = 6 & stop_trade_hour = 18 & tp_multiple = 5 DONE
range to consider = 5 & close range hour = 6 & stop_trade_hour = 18 & tp_multiple = 5.5 DONE


c:\Users\kyanh\AppData\Local\Programs\Python\Python313-arm64\Lib\site-packages\backtesting\backtesting.py:1009: UserWarning: time=29408: Broker canceled the order due to insufficient margin (equity=6672.25, margin_available=1209.57).
  warnings.warn(
c:\Users\kyanh\AppData\Local\Programs\Python\Python313-arm64\Lib\site-packages\backtesting\backtesting.py:1009: UserWarning: time=29435: Broker canceled the order due to insufficient margin (equity=6525.67, margin_available=1065.92).
  warnings.warn(


range to consider = 5 & close range hour = 6 & stop_trade_hour = 18 & tp_multiple = 6 DONE
range to consider = 5 & close range hour = 6 & stop_trade_hour = 19 & tp_multiple = 1 DONE
range to consider = 5 & close range hour = 6 & stop_trade_hour = 19 & tp_multiple = 1.5 DONE
range to consider = 5 & close range hour = 6 & stop_trade_hour = 19 & tp_multiple = 2 DONE


...

range to consider = 5 & close range hour = 9 & stop_trade_hour = 21 & tp_multiple = 5 DONE


c:\Users\kyanh\AppData\Local\Programs\Python\Python313-arm64\Lib\site-packages\backtesting\backtesting.py:1009: UserWarning: time=28377: Broker canceled the order due to insufficient margin (equity=2335.39, margin_available=1244.69).
  warnings.warn(
c:\Users\kyanh\AppData\Local\Programs\Python\Python313-arm64\Lib\site-packages\backtesting\backtesting.py:1009: UserWarning: time=28397: Broker canceled the order due to insufficient margin (equity=2446.55, margin_available=1353.62).
  warnings.warn(
c:\Users\kyanh\AppData\Local\Programs\Python\Python313-arm64\Lib\site-packages\backtesting\backtesting.py:1009: UserWarning: time=28454: Broker canceled the order due to insufficient margin (equity=2837.17, margin_available=420.74).
  warnings.warn(
c:\Users\kyanh\AppData\Local\Programs\Python\Python313-arm64\Lib\site-packages\backtesting\backtesting.py:1009: UserWarning: time=28478: Broker canceled the order due to insufficient margin (equity=3120.40, margin_available=698.30).
  warnings.warn

range to consider = 5 & close range hour = 9 & stop_trade_hour = 21 & tp_multiple = 5.5 DONE


c:\Users\kyanh\AppData\Local\Programs\Python\Python313-arm64\Lib\site-packages\backtesting\backtesting.py:1009: UserWarning: time=28377: Broker canceled the order due to insufficient margin (equity=2285.40, margin_available=1194.70).
  warnings.warn(
c:\Users\kyanh\AppData\Local\Programs\Python\Python313-arm64\Lib\site-packages\backtesting\backtesting.py:1009: UserWarning: time=28397: Broker canceled the order due to insufficient margin (equity=2396.56, margin_available=1303.63).
  warnings.warn(
c:\Users\kyanh\AppData\Local\Programs\Python\Python313-arm64\Lib\site-packages\backtesting\backtesting.py:1009: UserWarning: time=28454: Broker canceled the order due to insufficient margin (equity=2787.18, margin_available=370.75).
  warnings.warn(
c:\Users\kyanh\AppData\Local\Programs\Python\Python313-arm64\Lib\site-packages\backtesting\backtesting.py:1009: UserWarning: time=28478: Broker canceled the order due to insufficient margin (equity=3070.41, margin_available=648.31).
  warnings.warn

range to consider = 5 & close range hour = 9 & stop_trade_hour = 21 & tp_multiple = 6 DONE
